## Week 4 Search Eval
* Calculate `Hit Rate` (did the correct doc appear at all in top-5?) and `MRR` (Mean Reciprocal Rank, which rewards finding it earlier — rank 1 is best)

In [6]:
# Load the synthetic data created by 01_data_gen.ipynb notebook
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
df_ground_truth.head()

,question,document
0,I found this course late — can I still start i...,74eb249bbf
1,Is it okay to join the course after it has alr...,74eb249bbf
2,"If I enroll now, will I still be able to get a...",74eb249bbf
3,What’s the deadline for submitting the project...,74eb249bbf
4,"Can I join the course anytime, or do I have to...",74eb249bbf


In [7]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [1]:
from ingest import (
    load_faq_data,
    build_index,
)  # build index with minsearch (in-memory DB)

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

boost = {"question": 3.0}
index.search("what's this course for", num_results=3, boost_dict=boost)

[{'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'bd31146b0e',


In [2]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(query, num_results=5, boost_dict=boost_dict)

In [3]:
q = ground_truth[0]
q

NameError: name 'ground_truth' is not defined

In [ ]:
doc_id = q["document"]
results = text_search(query=q["question"])

# First, compare the retrieved document IDs with the correct document ID:
for d in results:
    print(f"{d['id']} == {doc_id}: {d['id'] == doc_id}")

0fab61eca2 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
04919992b3 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [ ]:
# calculate relevance metrics for retried docs
# This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document.

relevance = []
for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 0, 1, 0, 0]

In [4]:
# putting this into a function
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
q = ground_truth[10]
print(q["question"])
compute_relevance_text(q)

How do students join the Office Hours or live workshop if the Zoom link isn’t public?


[1, 0, 0, 0, 0]

In [ ]:
# calculate relevance matrix for first 15 docs
ground_truth_sample = ground_truth[:15]

from tqdm.auto import tqdm


def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total


relevance_total_text = compute_relevance_total_text(ground_truth_sample)
relevance_total_text

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

In [13]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

The total relevance function gets a search_function too.

We need to provide it explicitly:

In [14]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

Use it with `text_search` on the same sample:

In [15]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

Now run it for all ground truth questions:

In [16]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]